<a href="https://colab.research.google.com/github/rwcitek/TechEx-LangGraph-workshop/blob/main/notebooks/06_capstone_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — Optimization & Iteration

> **Time:** ~6 minutes hands-on (the lecture portion is the slides).
>
> **What you'll do:** pick **ONE** optimization, apply it, re-run the Module 5 eval suite, and compare the new aggregate to the baseline.

The system, dataset, and scorers below are identical to the Module 5 solution. Once you have baseline numbers, you'll edit ONE thing — prompt, workflow, model, or reliability — and measure whether it actually helps.

> 🏁  **Definition of done:** you can name your change, show before/after numbers, and explain *why* the numbers moved.

## 1.  Setup

In [ ]:
%pip install -q \
    langgraph==0.2.* \
    langchain==0.3.* \
    langchain-openai==0.2.*


In [ ]:
import os
from typing import TypedDict, Annotated, Literal
from operator import add
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
import json


In [ ]:
def _ensure_key(name: str, optional: bool = False) -> None:
    """Load an API key from (in order): existing env, Colab Secrets, or getpass.

    Colab Secrets are the recommended path for this workshop — set them ONCE
    via the 🔑 key icon in Colab's left sidebar and every notebook will pick
    them up automatically.
    """
    if os.environ.get(name):
        print(f"  ✓  {name} already set in environment")
        return
    # 1. Try Colab Secrets
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f"  ✓  {name} loaded from Colab Secrets")
            return
    except Exception:
        pass
    # 2. Fallback — prompt the user
    import getpass
    val = getpass.getpass(f"Paste your {name}{' (optional)' if optional else ''}: ").strip()
    if val:
        os.environ[name] = val
        print(f"  ✓  {name} set")
    elif optional:
        print(f"  •  {name} skipped (optional)")
    else:
        print(f"  ⚠   {name} skipped — you'll hit errors later without it")

_ensure_key("OPENAI_API_KEY")

print("Ready.")


## 2.  Baseline system (same as M5 solution)

In [ ]:
KB = {
    "billing": [
        {"id": "B-001", "title": "Billing cycle and prorations",
         "text": "We bill on the same day each month. If you change plans mid-cycle, the next invoice is prorated."},
        {"id": "B-002", "title": "Refund policy",
         "text": "Refunds are available within 14 days. Issued to original payment method, settle in 5 business days."},
        {"id": "B-003", "title": "Failed payments",
         "text": "Failed payments retry once a day for 3 days, then 7-day grace period."},
    ],
    "technical": [
        {"id": "T-001", "title": "Login issues",
         "text": "Clear cookies. For MFA failures check spam and verify phone."},
        {"id": "T-002", "title": "API rate limits",
         "text": "Standard plan: 60 requests per minute. Enterprise: 600. Respect Retry-After header."},
        {"id": "T-003", "title": "Export failures",
         "text": "Retry from the same dialog — export jobs are idempotent."},
    ],
    "account": [
        {"id": "A-001", "title": "Password reset",
         "text": "Use Forgot password. Reset email valid for 30 minutes. Check spam."},
        {"id": "A-002", "title": "Account closure",
         "text": "Settings → Account → Close. 30-day pending-deletion window."},
        {"id": "A-003", "title": "Ownership transfer",
         "text": "Owner invites new owner as Admin, both confirm via email."},
    ],
}


class TriageState(TypedDict):
    ticket: str
    category: Literal["billing", "technical", "account"] | None
    urgency:  Literal["low", "med", "high"] | None
    retrieved: list[dict]
    draft: str
    verdict: Literal["pass", "revise"] | None
    revision_count: int
    revisions: Annotated[list[str], add]


def make_initial_state(ticket):
    return {"ticket": ticket, "category": None, "urgency": None,
            "retrieved": [], "draft": "", "verdict": None,
            "revision_count": 0, "revisions": []}


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def parse_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"): text = text[4:]
    return json.loads(text.strip())


# --- Baseline agents (you'll modify ONE of these in the lab) ---

CLASSIFY_PROMPT = """\
Categorize the ticket as one of: billing, technical, account.
Rate urgency as: low, med, high.
Return JSON: {{"category": "...", "urgency": "..."}}

TICKET: {ticket}
"""

def classify(state):
    r = llm.invoke(CLASSIFY_PROMPT.format(ticket=state["ticket"]))
    p = parse_json(r.content)
    return {"category": p["category"], "urgency": p["urgency"]}


def retrieve(state):
    return {"retrieved": KB.get(state["category"], [])}


DRAFT_PROMPT = """\
You are a customer support agent.

Reply to the ticket using ONLY the policies below. If a policy doesn't
answer the question, say so. Do not invent details.

POLICIES:
{context}

TICKET: {ticket}

Write a concise reply (3-6 sentences). Cite the policy ID when used.
"""

def draft(state):
    ctx = "\n\n".join(f"[{d['id']}] {d['title']}\n{d['text']}"
                        for d in state["retrieved"])
    r = llm.invoke(DRAFT_PROMPT.format(context=ctx, ticket=state["ticket"]))
    return {"draft": r.content, "revisions": [r.content]}


QA_PROMPT = """\
Decide if the DRAFT is acceptable. Return JSON: {{"verdict": "pass" | "revise"}}
TICKET: {ticket}
DRAFT:  {draft}
"""

def qa(state):
    r = llm.invoke(QA_PROMPT.format(ticket=state["ticket"], draft=state["draft"]))
    p = parse_json(r.content)
    return {"verdict": p["verdict"], "revision_count": state["revision_count"] + 1}


MAX_REVISIONS = 2

def route_qa(state):
    if state["verdict"] == "pass": return "END"
    if state["revision_count"] >= MAX_REVISIONS: return "END"
    return "drafter"


def build_graph(classify_fn=classify, retrieve_fn=retrieve,
                draft_fn=draft, qa_fn=qa):
    g = StateGraph(TriageState)
    g.add_node("classify", classify_fn)
    g.add_node("retrieve", retrieve_fn)
    g.add_node("drafter",  draft_fn)
    g.add_node("qa",       qa_fn)
    g.set_entry_point("classify")
    g.add_edge("classify", "retrieve")
    g.add_edge("retrieve", "drafter")
    g.add_edge("drafter",  "qa")
    g.add_conditional_edges("qa", route_qa,
        {"drafter": "drafter", "END": END})
    return g.compile()


app = build_graph()
print("Baseline system ready.")

## 3.  Eval suite (same as M5 solution)

In [ ]:
EVAL_EXAMPLES = [
    {"id": "E-001", "inputs": {"ticket": "Hi, my monthly bill is $50 higher than last month. Can you check?"},
     "outputs": {"category": "billing", "urgency": "med",
                 "must_mention": ["proration"], "must_cite": ["B-001"]}},
    {"id": "E-002", "inputs": {"ticket": "Charged me twice for the same month. Refund please."},
     "outputs": {"category": "billing", "urgency": "high",
                 "must_mention": ["refund", "14 days"], "must_cite": ["B-002"]}},
    {"id": "E-003", "inputs": {"ticket": "I keep getting 429 errors from your API on the Standard plan."},
     "outputs": {"category": "technical", "urgency": "low",
                 "must_mention": ["60", "Retry-After"], "must_cite": ["T-002"]}},
    {"id": "E-004", "inputs": {"ticket": "Data export failed twice this morning."},
     "outputs": {"category": "technical", "urgency": "med",
                 "must_mention": ["retry", "idempotent"], "must_cite": ["T-003"]}},
    {"id": "E-005", "inputs": {"ticket": "Forgot my password and reset email never came. Demo in 30 minutes!"},
     "outputs": {"category": "account", "urgency": "high",
                 "must_mention": ["spam", "30 minutes"], "must_cite": ["A-001"]}},
    {"id": "E-006", "inputs": {"ticket": "What's the meaning of life? Also can you upgrade my plan to Enterprise?"},
     "outputs": {"category": "billing", "urgency": "low",
                 "must_mention": ["upgrade"], "must_cite": []}},
]


def score_category(run, ex):
    return {"key": "category_match",
            "score": int(run["outputs"]["category"] == ex["outputs"]["category"])}


def score_format(run, ex):
    d = run["outputs"]["draft"]
    leak = any(t in d for t in ["{ticket}", "{context}"])
    miss = [c for c in ex["outputs"].get("must_cite", []) if c not in d]
    return {"key": "format_ok", "score": int(not leak and not miss)}


JUDGE_PROMPT = """\
Score the REPLY from 1 (wrong) to 5 (correct, on-policy, helpful).
Return JSON: {{"score": <int>, "reason": "..."}}

TICKET: {ticket}
SHOULD MENTION: {must_mention}
REPLY: {draft}
"""

def judge_helpfulness(run, ex):
    prompt = JUDGE_PROMPT.format(
        ticket=ex["inputs"]["ticket"],
        must_mention=", ".join(ex["outputs"].get("must_mention", [])),
        draft=run["outputs"]["draft"],
    )
    r = llm.invoke(prompt)
    p = parse_json(r.content)
    return {"key": "helpfulness", "score": p["score"] / 5.0}


SCORERS = [score_category, score_format, judge_helpfulness]


def run_eval(app, examples, scorers):
    rows = []
    for ex in examples:
        state = app.invoke(make_initial_state(ex["inputs"]["ticket"]))
        run = {"outputs": state}
        for s in scorers:
            row = s(run, ex)
            rows.append({"example": ex["id"], **row})
    return rows


def aggregate(rows):
    by = {}
    for r in rows:
        by.setdefault(r["key"], []).append(r["score"])
    return {k: sum(v) / len(v) for k, v in by.items()}


print(f"Loaded {len(EVAL_EXAMPLES)} examples, {len(SCORERS)} scorers.")

## 4.  Baseline run

In [ ]:
print("Running baseline...")
baseline_rows = run_eval(app, EVAL_EXAMPLES, SCORERS)
baseline = aggregate(baseline_rows)

for k, v in baseline.items():
    print(f"  baseline   {k:>16s}  =  {v:.3f}")

## 5.  The optimization menu

Pick **ONE** quadrant. Apply ONE change. Re-run.

| Quadrant         | Examples                                                                      |
|------------------|-------------------------------------------------------------------------------|
| 🟦 Prompt-level    | Add few-shot examples to Classifier · stricter format in DRAFT_PROMPT          |
| 🟪 Workflow-level  | Skip QA when classify says urgency=low · cache retrieved docs per category     |
| 🟨 Model-level     | Cheaper model for Classifier/QA · keep Drafter on better model                 |
| 🟩 Reliability     | Retry on transient errors with `tenacity` · validate JSON before parsing       |

Below are four stubs — pick one and fill it in. Leave the others untouched.

### 🟦  Option A — Prompt-level

In [ ]:
# 🟦  OPTION A — Prompt-level: add few-shot examples to Classifier
#
# Currently CLASSIFY_PROMPT has zero examples. Add 2-3 worked examples (one per
# category) directly in the prompt to anchor the model on the output shape.
#
# Edit CLASSIFY_PROMPT below if you pick this option, then redefine `classify`.

CLASSIFY_PROMPT_V2 = """\
Categorize the ticket as one of: billing, technical, account.
Rate urgency as: low, med, high.
Return JSON: {{"category": "...", "urgency": "..."}}

# TODO — add 2-3 worked examples here. Format each as:
#   TICKET: <example ticket>
#   {{"category": "<cat>", "urgency": "<urg>"}}
# Cover one billing, one technical, one account case.

EXAMPLES:
TICKET: <example ticket>
{{"category": "...", "urgency": "..."}}

TICKET: {ticket}
"""

# Don't redefine `classify` yet — uncomment when you've added the examples.
# def classify_v2(state):
#     r = llm.invoke(CLASSIFY_PROMPT_V2.format(ticket=state["ticket"]))
#     p = parse_json(r.content)
#     return {"category": p["category"], "urgency": p["urgency"]}

### 🟪  Option B — Workflow-level

In [ ]:
# 🟪  OPTION B — Workflow-level: skip QA when urgency=low
#
# QA adds ~1s of latency and a token cost. For very-low-urgency tickets we
# can ship the first draft directly.
#
# To do this, edit `route_qa` (or replace it) so that when state["urgency"]
# == "low", we skip QA entirely. Or restructure the graph to route around QA
# for low-urgency. Either approach works — pick whichever feels cleaner.

# TODO — write your modified routing here, or restructure the graph.

# def route_qa_v2(state):
#     ...

### 🟨  Option C — Model-level

In [ ]:
# 🟨  OPTION C — Model-level: cheaper model for Classifier and QA
#
# Classifier and QA do simple decision tasks. We can use a smaller / cheaper
# model for these and keep gpt-4o-mini only for Drafter (or use gpt-4o for
# Drafter for higher quality).
#
# Hint: ChatOpenAI(model="gpt-4o-mini") is already pretty cheap.
# If your account has it, try gpt-3.5-turbo for Classifier and QA. If not,
# create a second instance with a different temperature or use streaming.

# TODO — define new LLM instances and redefine classify / qa to use them.

# llm_tiny = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
# def classify_v2(state): ...
# def qa_v2(state): ...

### 🟩  Option D — Reliability-level

In [ ]:
# 🟩  OPTION D — Reliability: validate JSON, retry on transient errors
#
# parse_json() raises on bad output. In production we'd retry the call.
# Wrap the classifier (or QA) with a retry helper.
#
# Hint: a simple try/except + one retry is enough for the workshop.

# TODO — write a retry helper and apply it.

# def with_one_retry(fn):
#     def wrapped(state):
#         try:
#             return fn(state)
#         except Exception:
#             return fn(state)
#     return wrapped
#
# classify_v2 = with_one_retry(classify)

## 6.  Apply your change and re-eval

In [ ]:
# Once you've defined classify_v2 (or whichever function you replaced),
# build a new app using the v2 versions and re-run eval.

# TODO — build the new app
# improved_app = build_graph(classify_fn=classify_v2)   # or whichever you replaced

# Uncomment when ready:
# print("Running improved candidate...")
# improved_rows = run_eval(improved_app, EVAL_EXAMPLES, SCORERS)
# improved = aggregate(improved_rows)
# for k, v in improved.items():
#     print(f"  improved  {k:>16s}  =  {v:.3f}")

## 7.  Compare before / after

In [ ]:
print(f"  {'scorer':>16s}   baseline   improved   delta")
print("  " + "-" * 52)
for k in sorted(baseline):
    delta = improved[k] - baseline[k]
    mark = "✓" if delta >= 0 else "⚠"
    print(f"  {k:>16s}   {baseline[k]:.3f}     {improved[k]:.3f}    {delta:+.3f}  {mark}")

# Also count: how many examples flipped status on the helpfulness scorer?
def per_example_score(rows, key):
    return {r["example"]: r["score"] for r in rows if r["key"] == key}

base_help = per_example_score(baseline_rows, "helpfulness")
new_help  = per_example_score(improved_rows, "helpfulness")

print(f"\n  example-level helpfulness changes:")
for e in sorted(base_help):
    delta = new_help[e] - base_help[e]
    arrow = "▲" if delta > 0.05 else "▼" if delta < -0.05 else "—"
    print(f"    {e}:  {base_help[e]:.2f}  →  {new_help[e]:.2f}   {arrow}")

## 8.  Write up your change

Fill in the dict below. This is the *real* deliverable of the capstone — a small artifact you can paste into a PR description tomorrow.

In [ ]:
CAPSTONE = {
    "quadrant":      "...",   # 'prompt' / 'workflow' / 'model' / 'reliability'
    "change":        "...",   # one sentence describing what you edited
    "metric_moved":  "...",   # which scorer changed?
    "delta":          0.0,    # by how much (use the deltas above)
    "regressed_anything":  False,
    "would_ship":    False,
    "why":           "...",
}
print(CAPSTONE)

## Wrap up — and that's the workshop

You came in able to *prototype* an agent. You're leaving able to *ship* one.

Recap of what's in your toolkit now:

1. **Design** — typed shared state, explicit topology, termination guards
2. **Build** — LangGraph primitives, the four-layer agent anatomy
3. **Debug** — three failure families + the reproduce-isolate-fix loop
4. **Observe** — traces, metadata, the bottleneck-finding workflow
5. **Evaluate** — rule-based + LLM-as-judge scorers, regression catching
6. **Optimize** — pick one quadrant, measure, repeat

Take this notebook home. Swap in your own knowledge base. Swap in your own use case. Ship it.

Thank you for spending the day.